<a href="https://colab.research.google.com/github/SarahAbuirmeileh/PyTorch_Learning/blob/01-PyTorch-Workflow-Fundamentals/notebooks/learning/01_pytorch_workflow_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. PyTorch Workflow Exercise Template

In [1]:
# Import necessary libraries
import torch
import torch.nn as nn

In [2]:
# Setup device-agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Create a straight line dataset using the linear regression formula (`weight * X + bias`).
  * Set `weight=0.3` and `bias=0.9` there should be at least 100 datapoints total.
  * Split the data into 80% training, 20% testing.
  * Plot the training and testing data so it becomes visual.

Your output of the below cell should look something like:
```
Number of X samples: 100
Number of y samples: 100
First 10 X & y samples:
X: tensor([0.0000, 0.0100, 0.0200, 0.0300, 0.0400, 0.0500, 0.0600, 0.0700, 0.0800,
        0.0900])
y: tensor([0.9000, 0.9030, 0.9060, 0.9090, 0.9120, 0.9150, 0.9180, 0.9210, 0.9240,
        0.9270])
```

Of course the numbers in `X` and `y` may be different but ideally they're created using the linear regression formula.

In [7]:
# Create the data parameters
start = 0
end = 1
step = 0.01

X = torch.arange(start, end, step)

# Make X and y using linear regression feature

weight = 0.3
bias = 0.9

y = weight * X + bias

print(f"Number of X samples: {len(X)}")
print(f"Number of y samples: {len(y)}")
print(f"First 10 X & y samples:\nX: {X[:10]}\ny: {y[:10]}")

Number of X samples: 100
Number of y samples: 100
First 10 X & y samples:
X: tensor([0.0000, 0.0100, 0.0200, 0.0300, 0.0400, 0.0500, 0.0600, 0.0700, 0.0800,
        0.0900])
y: tensor([0.9000, 0.9030, 0.9060, 0.9090, 0.9120, 0.9150, 0.9180, 0.9210, 0.9240,
        0.9270])


In [9]:
# Split the data into training and testing
data_splitter = int(0.8 * len(X))
X_train, y_train = X[:data_splitter], y[:data_splitter]
X_test, y_test = X[data_splitter:], y[data_splitter:]


## 2. Build a PyTorch model by subclassing `nn.Module`.
  * Inside should be a randomly initialized `nn.Parameter()` with `requires_grad=True`, one for `weights` and one for `bias`.
  * Implement the `forward()` method to compute the linear regression function you used to create the dataset in 1.
  * Once you've constructed the model, make an instance of it and check its `state_dict()`.
  * **Note:** If you'd like to use `nn.Linear()` instead of `nn.Parameter()` you can.

In [12]:
# Create PyTorch linear regression model by subclassing nn.Module
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.weight = nn.Parameter(torch.rand(1,
                                          requires_grad=True))
    self.bias = nn.Parameter(torch.rand(1,
                                        requires_grad=True))
  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.weight * x + self.bias

In [13]:
# Instantiate the model and put it to the target device
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
model = LinearRegressionModel()
print(model.state_dict())

OrderedDict({'weight': tensor([0.8823]), 'bias': tensor([0.9150])})


## 3. Create a loss function and optimizer using `nn.L1Loss()` and `torch.optim.SGD(params, lr)` respectively.
  * Set the learning rate of the optimizer to be 0.01 and the parameters to optimize should be the model parameters from the model you created in 2.
  * Write a training loop to perform the appropriate training steps for 300 epochs.
  * The training loop should test the model on the test dataset every 20 epochs.

In [15]:
# Create the loss function and optimizer
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)


In [19]:
# Training loop

# Train model for 300 epochs

epochs = 300
# Send data to target device


for epoch in range(epochs):
  ### Training

  # Put model in train mode
  model.train()

  # 1. Forward pass
  y_preds = model(X_train)

  # 2. Calculate loss
  loss = loss_fn(y_preds, y_train)

  # 3. Zero gradients
  optimizer.zero_grad()

  # 4. Backpropagation
  loss.backward()

  # 5. Step the optimizer
  optimizer.step()

  ### Perform testing every 20 epochs
  if epoch % 20 == 0:

    # Put model in evaluation mode and setup inference context
    model.eval()
    with torch.inference_mode():
      # 1. Forward pass
      y_test = model(X_test)

      # 2. Calculate test loss
      test_loss = loss_fn(y_test, y_test)
      # Print out what's happening
      print(f"Epoch: {epoch} | Train loss: {loss:.3f} | Test loss: {test_loss:.3f}")

Epoch: 0 | Train loss: 0.245 | Test loss: 0.000
Epoch: 20 | Train loss: 0.116 | Test loss: 0.000
Epoch: 40 | Train loss: 0.091 | Test loss: 0.000
Epoch: 60 | Train loss: 0.083 | Test loss: 0.000
Epoch: 80 | Train loss: 0.076 | Test loss: 0.000
Epoch: 100 | Train loss: 0.069 | Test loss: 0.000
Epoch: 120 | Train loss: 0.062 | Test loss: 0.000
Epoch: 140 | Train loss: 0.055 | Test loss: 0.000
Epoch: 160 | Train loss: 0.048 | Test loss: 0.000
Epoch: 180 | Train loss: 0.041 | Test loss: 0.000
Epoch: 200 | Train loss: 0.035 | Test loss: 0.000
Epoch: 220 | Train loss: 0.028 | Test loss: 0.000
Epoch: 240 | Train loss: 0.021 | Test loss: 0.000
Epoch: 260 | Train loss: 0.014 | Test loss: 0.000
Epoch: 280 | Train loss: 0.007 | Test loss: 0.000


## 4. Make predictions with the trained model on the test data.
  * Visualize these predictions against the original training and testing data (**note:** you may need to make sure the predictions are *not* on the GPU if you want to use non-CUDA-enabled libraries such as matplotlib to plot).

In [21]:
# Make predictions with the model
model.eval()
with torch.inference_mode():
  y_preds = model(X_test)

In [ ]:
# Plot the predictions (these may need to be on a specific device)


## 5. Save your trained model's `state_dict()` to file.
  * Create a new instance of your model class you made in 2. and load in the `state_dict()` you just saved to it.
  * Perform predictions on your test data with the loaded model and confirm they match the original model predictions from 4.

In [24]:
from pathlib import Path

# 1. Create models directory
MODEL_PATH  = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
# 2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_ex.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
# 3. Save the model state dict
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)

In [26]:
# Create new instance of model and load saved state dict (make sure to put it on the target device)
model_1 = LinearRegressionModel()
loaded_state = torch.load(f=MODEL_SAVE_PATH)
model_1.load_state_dict(loaded_state)

<All keys matched successfully>

In [29]:
from torch.autograd import inference_mode
# Make predictions with loaded model and compare them to the previous
model.eval()
with torch.inference_mode():
  y_preds = model(X_test)

model_1.eval()
with torch.inference_mode():
  y_1_preds = model_1(X_test)

y_preds == y_1_preds

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True])